In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import pandas as pd
import numpy as np

# =========================================================================================
# 📚 [튜터 가이드]
# 데이터셋 이름: maywell/korean_textbooks
# 데이터셋 의미: 대규모 한국어 합성 텍스트 데이터셋 (Korean Synthetic Textbook Data)
# 설명: 이 데이터셋은 다양한 주제의 교과서 및 텍스트 내용을 모아 놓은 대규모 한국어 텍스트입니다.
# 즉, 한국어 AI가 문맥을 이해하고 지식을 습득하는 데 사용하는 '재료'라고 생각하시면 됩니다.
# 이 데이터로 무엇을 할 수 있을까요? -> 단순한 문장을 넘어, '지식 구조'나 '주제별 패턴'을 뽑아내는 연습을 할 수 있습니다!
# =========================================================================================

# --- 설정 상수 ---
DATASET_NAME = "maywell/korean_textbooks"
SAMPLE_COUNT = 50 # 실습 시 분석할 샘플 개수 (너무 많으면 로딩 시간이 길어집니다!)

def load_korean_dataset():
    """
    데이터셋을 스트리밍 모드와 일반 모드를 모두 시도하며 로드하는 함수입니다.
    스트리밍이 실패할 경우 일반 다운로드 모드로 전환합니다.
    """
    print("✨ 튜터: 데이터셋을 로드할 준비를 하고 있어요. 가장 빠르게 학습할 수 있는 '스트리밍' 방식을 먼저 시도해볼게요!")
    dataset = None
    try:
        # 1. 스트리밍 모드 (Memory 효율적, 대용량 데이터에 최적!)
        dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
        print("✅ 성공! 스트리밍 모드로 데이터셋을 로드했습니다. 메모리 걱정 없이 큰 데이터를 다룰 수 있어요!")
        return dataset
    except Exception as e:
        print(f"⚠️ 경고: 스트리밍 로드에 실패했습니다 ({e}). 일반 다운로드 모드(streaming=False)로 전환하여 진행할게요.")
        try:
            # 2. 일반 모드 (streaming=False, 실제 데이터를 다운로드)
            return load_dataset(DATASET_NAME, split='train', streaming=False)
        except Exception as e:
            print(f"❌ 치명적 오류: 데이터셋 로드에 실패했습니다. ({e})")
            return None

# 1. 데이터 로드 (가장 중요한 단계!)
dataset = load_korean_dataset()

if dataset is None:
    print("\n😭 데이터셋을 불러오는 데 실패하여 실습을 중단합니다.")
    exit()


# 2. 샘플 데이터 추출 (전체를 다룰 필요는 없어요!)
print("\n🚀 데이터셋에서 분석할 샘플 100개를 골라와요...")

# 🎁 필수 패턴 적용: .take()를 사용하여 샘플 데이터셋 이터레이터를 만듭니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    # iterator 패턴을 사용하기 위해 list(dataset.take(K))를 사용합니다.
    sample_list = list(dataset.take(SAMPLE_COUNT))
    sample_dataset_iterator = iter(sample_list)
    print(f"✅ 성공! 총 {len(sample_list)}개의 샘플을 추출했습니다.")
else:
    # 일반 데이터셋 (Dataset) - 이 경우 list() 변환만으로 충분합니다.
    sample_list = list(dataset.take(SAMPLE_COUNT))
    sample_dataset_iterator = iter(sample_list)


# 3. 🧑‍💻 초보자를 위한 창의적 실습: '지식 문맥 요약 및 키워드 추출' 시뮬레이션
# 목표: 데이터가 너무 길 때, 핵심 문맥을 빠르게 찾고 요약하는 능력을 키워봅시다.
# 실제 LLM(대형 언어 모델)의 역할을 흉내 내보는 재미있는 연습이에요!

def analyze_sample(sample_iterator):
    """
    샘플 데이터셋을 순회하며 텍스트를 분석하고, 재미있는 패턴을 뽑아냅니다.
    """
    print("\n🌟 실습 시작: '지식 패턴 탐색기' 가동!")
    
    # 데이터셋의 첫 샘플을 준비합니다.
    try:
        sample_data = next(sample_iterator)['text']
    except StopIteration:
        print("🚨 샘플 데이터가 없습니다. 빈 데이터셋일 수 있습니다.")
        return
    
    print("\n==================================================================================")
    print("✨ [실습 예시 1: 핵심 주제 식별하기] - 문장이 어떤 분야인지 추측해보기")
    print("----------------------------------------------------------------------------------")
    print(f"🔍 분석 원문 (Sample 1): {sample_data[:100]}...")

    # 1. 길이 분석 및 감점 체크 (간단한 정량적 분석)
    word_count = len(sample_data)
    print(f"📏 분석 결과: 텍스트 길이는 총 {word_count} 글자입니다.")

    # 2. 창의적 로직: 텍스트에 '물리학' 또는 '역사' 같은 특정 키워드가 있는지 체크
    if "원리" in sample_data or "에너지" in sample_data or "입자" in sample_data:
        theme = "💡 물리학/과학 이론 관련 텍스트로 추정됩니다!"
    elif "시대" in sample_data or "왕조" in sample_data or "역사적" in sample_data:
        theme = "🕰️ 역사 또는 사회 구조 관련 내용으로 보여요!"
    elif "방법" in sample_data or "절차" in sample_data:
        theme = "📝 지침서나 절차(How-to)에 대한 내용일 가능성이 높습니다."
    else:
        theme = "📚 일반적인 교과서의 설명문이거나 다른 주제일 수 있습니다."
    
    print(f"✅ 추론된 주제: {theme}")
    print("----------------------------------------------------------------------------------")


    print("\n✨ [실습 예시 2: 문장 구조 분해하기] - 가장 긴 문장과 핵심 키워드 찾기")
    print("----------------------------------------------------------------------------------")
    
    # 3. 실습: 마침표(.) 개수를 세어 문장의 복잡성을 가늠하기
    period_count = sample_data.count('.')
    print(f"🔍 문장 구조 분석: 원문에는 {period_count}개의 마침표가 발견됩니다.")
    
    # 마침표 개수와 텍스트 길이를 이용한 단순 패턴화
    if period_count > 5 and word_count > 50:
        print("✨ 튜터 예측: 이 텍스트는 여러 개의 세부 주제를 연결하는 '요약/정리' 형식의 텍스트일 확률이 높아요!")
    elif period_count <= 1 and word_count > 100:
        print("✨ 튜터 예측: 단일 주제를 깊게 파고드는 '설명문' 형식의 텍스트일 확률이 높습니다.")
    else:
        print("✨ 튜터 예측: 균형 잡힌 서술형 설명문으로 보입니다.")
    
    print("==================================================================================")
    print("✨ 분석 끝! 이처럼 텍스트 데이터를 분석할 때, 단순한 LLM 추론 외에도 길이, 패턴, 키워드 같은 정량적 분석을 병행하면 AI의 성능을 높일 수 있습니다! 👏")


# 4. 실행
analyze_sample(sample_dataset_iterator)